In [43]:
import pandas as pd
import numpy as np
import os

In [44]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("s1nghayush/ig-filter-data")

print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/s1nghayush/ig-filter-data/versions/2


Converting BBB to MBM

In [45]:
df = pd.read_csv(path + "/ganjoo_ipl_bbb.csv")

In [46]:
df.drop(columns=['wagonX', 'wagonY', 'wagonZone', 'line','length', 'shot', 'control', 'predscore', 'wprob'], inplace=True)

In [ ]:
df.columns

Index(['p_match', 'inns', 'bat', 'p_bat', 'team_bat', 'bowl', 'p_bowl',
       'team_bowl', 'ball', 'ball_id', 'outcome', 'score', 'out', 'dismissal',
       'p_out', 'over', 'noball', 'wide', 'byes', 'legbyes', 'cur_bat_runs',
       'cur_bat_bf', 'cur_bowl_ovr', 'cur_bowl_wkts', 'cur_bowl_runs',
       'inns_runs', 'inns_wkts', 'inns_balls', 'inns_runs_rem',
       'inns_balls_rem', 'inns_rr', 'inns_rrr', 'target', 'max_balls', 'date',
       'year', 'ground', 'country', 'winner', 'toss', 'competition',
       'bat_hand', 'bowl_style', 'bowl_kind', 'batruns', 'ballfaced',
       'bowlruns', 'bat_out'],
      dtype='object')

In [ ]:
print(df.sample().iloc[0])

p_match                            1136597
inns                                     2
bat                             Chris Lynn
p_bat                               326637
team_bat             Kolkata Knight Riders
bowl                  Mitchell McClenaghan
p_bowl                              319439
team_bowl                   Mumbai Indians
ball                                     4
ball_id                               0.04
outcome                             no run
score                                    0
out                                  False
dismissal                              NaN
p_out                               326637
over                                     1
noball                                   0
wide                                     0
byes                                     0
legbyes                                  0
cur_bat_runs                             1
cur_bat_bf                               3
cur_bowl_ovr                           0.4
cur_bowl_wk

In [110]:
import pandas as pd

def process_batting_scorecard_from_kaggle(df):
    """
    Process a ball-by-ball DataFrame from the Kaggle dataset into a batting scorecard format.
    It extracts team information (bat_team, bowl_team) and venue (ground) directly from the input data.

    Parameters:
      df (DataFrame): The ball-by-ball match data, assumed to be pre-filtered.

    Returns:
      DataFrame: The aggregated batting scorecard with:
               - bat_team: Batting team (from team_bat column)
               - bowl_team: Bowling team (from team_bowl column)
               - bowl_id: For dismissed batsmen, the bowler's ID taken from p_out.
               - venue: The ground (from ground column) used as the match venue.
    """
    if df.empty:
        return pd.DataFrame()

    # Capture the match date and venue (assuming all rows share the same values)
    match_date = df['date'].iloc[0] if 'date' in df.columns else ""
    venue = df['ground'].iloc[0] if 'ground' in df.columns else ""

    # Initialize batting statistics dictionary.
    # Key: (match_id, batting_team, striker)
    batting_stats = {}

    # Process each ball in the DataFrame.
    for _, ball in df.iterrows():
        # Convert p_bat to numeric.
        striker = pd.to_numeric(ball['p_bat'], errors='coerce')

        batsman_name = ball['bat']        # Batsman's name
        bowler_name = ball['bowl']          # Bowler's name for the current ball
        batting_team = ball['team_bat']     # Batting team name
        bowling_team = ball['team_bowl']    # Bowling team name
        match_id = ball['p_match']          # Match ID

        key = (match_id, batting_team, striker)
        if key not in batting_stats:
            # Initialize statistics.
            batting_stats[key] = {
                'batsman_name': batsman_name,
                'bowler_name': "",   # Set only when a wicket occurs.
                'runs': 0,
                'balls': 0,
                'fours': 0,
                'sixes': 0,
                'isOut': False,
                'wicketType': '',
                'innings': ball['inns'],
                'p_bat_id': striker,   # Raw striker ID.
                'p_bowl_id': None,     # Will be set when a wicket falls.
                'bat_team': batting_team,
                'bowl_team': bowling_team
            }
        stats = batting_stats[key]

        # Update runs and balls faced.
        runs = ball['batruns']
        stats['balls'] += 1
        stats['runs'] += runs

        # Count boundaries.
        if runs == 4:
            stats['fours'] += 1
        elif runs == 6:
            stats['sixes'] += 1
        elif runs == 5:
            stats['fours'] += 1  # Count a 5 as a four (with an extra wide).

        # Check if the batsman is dismissed on this ball.
        out_status = str(ball['out']).strip().upper()  # Normalize
        dismissed_player = str(ball['p_out']).strip()
        if out_status == 'TRUE' and dismissed_player == str(striker):
            stats['isOut'] = True
            stats['wicketType'] = str(ball['dismissal']).strip()
            stats['bowler_name'] = bowler_name  # Assign bowler only on dismissal.
            # Get bowler's id from p_out.
            stats['p_bowl_id'] = pd.to_numeric(ball['p_out'], errors='coerce')

        batting_stats[key] = stats

    # Create the results DataFrame with aggregated statistics.
    results = []
    for (match_id, team, player), stats in batting_stats.items():
        strike_rate = (stats['runs'] / stats['balls'] * 100) if stats['balls'] > 0 else 0
        results.append({
            'MatchID': match_id,
            'date': match_date,
            'venue': venue,  # Include the venue (ground)
            'innings': stats['innings'],
            'batsman': player,
            'batsman_name': stats['batsman_name'],
            'bowler_name': stats['bowler_name'],
            'bowl_id': stats['p_bowl_id'],  # Bowler's id from p_out when dismissal occurred.
            'runs': stats['runs'],
            'balls': stats['balls'],
            'fours': stats['fours'],
            'sixes': stats['sixes'],
            'strikeRate': round(strike_rate, 2),
            'isOut': stats['isOut'],
            'wicketType': stats['wicketType'] if stats['isOut'] else 'not out',
            'bat_team': stats['bat_team'],   # Batting team (from team_bat)
            'bowl_team': stats['bowl_team']  # Bowling team (from team_bowl)
        })

    return pd.DataFrame(results)


In [125]:
batting_card=process_batting_scorecard_from_kaggle(df)

In [127]:
batting_card.columns

Index(['MatchID', 'date', 'venue', 'innings', 'batsman', 'batsman_name',
       'bowler_name', 'bowl_id', 'runs', 'balls', 'fours', 'sixes',
       'strikeRate', 'isOut', 'wicketType', 'bat_team', 'bowl_team'],
      dtype='object')

In [128]:
batting_card.sort_values(by=['date', 'MatchID', 'innings'], inplace=True)

In [129]:
batting_card = batting_card.rename(columns={'MatchID': 'match_id','innings':'inns','batsman':'batter_id','batsman_name':'batter','bowler_name':'bowler','bowl_id':'bowler_id','strikeRate':'sr'})

In [130]:
batting_card = batting_card.reindex(columns=['date', 'match_id', 'bat_team', 'bowl_team', 'inns','batter_id','batter','runs','balls','fours','sixes','sr','isOut','wicketType','bowler_id', 'bowler','venue'])

In [131]:
batting_card['venue'] = batting_card['venue'].str.split(',').str.get(0)


In [132]:
batting_card.sample()

,date,match_id,bat_team,bowl_team,inns,batter_id,batter,runs,balls,fours,sixes,sr,isOut,wicketType,bowler_id,bowler,venue
363,2017-04-29,1082640,Gujarat Lions,Delhi Capitals,1,234675,Ravindra Jadeja,13,7,2,0,185.71,False,not out,NaN,,Saurashtra Cricket Association Stadium


In [133]:
batting_card[batting_card['match_id'] == 829705]

,date,match_id,bat_team,bowl_team,inns,batter_id,batter,runs,balls,fours,sixes,sr,isOut,wicketType,bowler_id,bowler,venue
466,2017-04-29,829705,Mumbai Indians,Kolkata Knight Riders,1,277662,Corey Anderson,55,41,4,3,134.15,False,not out,NaN,,Saurashtra Cricket Association Stadium
519,2017-04-29,829705,Mumbai Indians,Kolkata Knight Riders,1,34102,Rohit Sharma,98,66,12,4,148.48,False,not out,NaN,,Saurashtra Cricket Association Stadium
4385,2017-04-29,829705,Mumbai Indians,Kolkata Knight Riders,1,333904,Aditya Tare,7,8,1,0,87.50,True,caught,333904.0,Shakib Al Hasan,Saurashtra Cricket Association Stadium
5701,2017-04-29,829705,Mumbai Indians,Kolkata Knight Riders,1,5334,Aaron Finch,5,5,1,0,100.00,True,caught,5334.0,Morne Morkel,Saurashtra Cricket Association Stadium
6805,2017-04-29,829705,Mumbai Indians,Kolkata Knight Riders,1,33141,Ambati Rayudu,0,2,0,0,0.00,True,caught,33141.0,Morne Morkel,Saurashtra Cricket Association Stadium
19,2017-04-29,829705,Kolkata Knight Riders,Mumbai Indians,2,290630,Manish Pandey,40,24,2,3,166.67,True,caught,290630.0,Harbhajan Singh,Saurashtra Cricket Association Stadium
138,2017-04-29,829705,Kolkata Knight Riders,Mumbai Indians,2,28763,Gautam Gambhir,57,44,7,1,129.55,True,caught,28763.0,Jasprit Bumrah,Saurashtra Cricket Association Stadium
157,2017-04-29,829705,Kolkata Knight Riders,Mumbai Indians,2,35582,Robin Uthappa,9,12,0,1,75.00,True,caught,35582.0,Corey Anderson,Saurashtra Cricket Association Stadium
159,2017-04-29,829705,Kolkata Knight Riders,Mumbai Indians,2,32498,Yusuf Pathan,14,12,1,1,116.67,False,not out,NaN,,Saurashtra Cricket Association Stadium
163,2017-04-29,829705,Kolkata Knight Riders,Mumbai Indians,2,446507,Suryakumar Yadav,46,21,1,5,219.05,False,not out,NaN,,Saurashtra Cricket Association Stadium


In [135]:
batting_card.to_csv('ganjoo_ipl_mbm_bat.csv', index=False)